# Build patient history JSONs from MIMIC-IV + MIMIC-CXR-RRG

Produces one `patient_<subject_id>_history.json` per requested subject and a shared `cxrs/` folder of saved chest-X-ray JPEGs. The Streamlit app and `../src/make_vdbs.py` will pick up every JSON in this directory automatically.

Edit `SUBJECT_IDS` to the patients you want, then run all cells.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from dateutil.relativedelta import relativedelta
from tqdm import tqdm

In [ ]:
DATA_PATH = '../../physionet.org/files/mimiciv/3.1'
DATA_PATH_NOTES = '../../physionet.org/files/mimic-iv-note/2.2'
CXR_DATASET = '../../mimic-cxr'  # or 'Yamini-1628/MIMIC-CXR-RRG' to pull from the Hub
OUTPUT_DIR = Path('.')
CXRS_DIR = OUTPUT_DIR / 'cxrs'
CXRS_DIR.mkdir(parents=True, exist_ok=True)

# Patients to extract.
SUBJECT_IDS = [13221453]

In [ ]:
patients = pd.read_csv(f'{DATA_PATH}/hosp/patients.csv.gz')
admissions = pd.read_csv(f'{DATA_PATH}/hosp/admissions.csv.gz')
diagnoses = pd.read_csv(f'{DATA_PATH}/hosp/diagnoses_icd.csv.gz')
d_icd_diag = pd.read_csv(f'{DATA_PATH}/hosp/d_icd_diagnoses.csv.gz')
procedures = pd.read_csv(f'{DATA_PATH}/hosp/procedures_icd.csv.gz')
d_icd_proc = pd.read_csv(f'{DATA_PATH}/hosp/d_icd_procedures.csv.gz')
prescriptions = pd.read_csv(f'{DATA_PATH}/hosp/prescriptions.csv.gz', low_memory=False)
labevents = pd.read_csv(f'{DATA_PATH}/hosp/labevents.csv.gz')
d_labitems = pd.read_csv(f'{DATA_PATH}/hosp/d_labitems.csv.gz')
discharge_notes = pd.read_csv(f'{DATA_PATH_NOTES}/note/discharge.csv.gz')

In [ ]:
DATE_SHIFT = relativedelta(years=-126, months=8)

def build_history(subject_id: int) -> dict:
    pat_row = patients[patients.subject_id == subject_id]
    if pat_row.empty:
        raise ValueError(f'subject {subject_id} not found')
    patient = pat_row.iloc[0]
    patient_adm = admissions[admissions.subject_id == subject_id]

    history = {
        'subject_id': subject_id,
        'demographics': {
            'sex': patient['gender'],
            'anchor_age': int(patient['anchor_age']),
        },
        'admissions': [],
    }

    for _, adm in tqdm(patient_adm.iterrows(), total=len(patient_adm), desc=f'subject {subject_id}'):
        hadm_id = adm['hadm_id']

        dx = diagnoses[diagnoses.hadm_id == hadm_id].merge(d_icd_diag, on=['icd_code', 'icd_version'], how='left')
        dx_list = dx['long_title'].dropna().unique().tolist()

        proc = procedures[procedures.hadm_id == hadm_id].merge(d_icd_proc, on=['icd_code', 'icd_version'], how='left')
        proc_list = proc['long_title'].dropna().unique().tolist()

        meds = prescriptions[prescriptions.hadm_id == hadm_id]
        med_list = meds['drug'].dropna().unique().tolist()

        labs = labevents[labevents.hadm_id == hadm_id].merge(d_labitems, on='itemid', how='left')
        lab_summary = {}
        for labname, group in labs.groupby('label'):
            if len(lab_summary) > 10:
                break
            numeric = pd.to_numeric(group['valuenum'], errors='coerce').dropna()
            if len(numeric) > 0:
                lab_summary[labname] = {'min': float(numeric.min()), 'max': float(numeric.max())}

        note = discharge_notes[discharge_notes.hadm_id == hadm_id]
        discharge_text = note.iloc[0]['text'] if len(note) > 0 else ''

        adm_dt = datetime.strptime(adm['admittime'], '%Y-%m-%d %H:%M:%S')
        dis_dt = datetime.strptime(adm['dischtime'], '%Y-%m-%d %H:%M:%S')

        history['admissions'].append({
            'hadm_id': int(hadm_id),
            'admittime': str(adm_dt + DATE_SHIFT),
            'dischtime': str(dis_dt + DATE_SHIFT),
            'diagnoses': dx_list,
            'procedures': proc_list,
            'medications': med_list,
            'lab_summary': lab_summary,
            'discharge_summary': discharge_text,
        })

    return history

In [ ]:
cxr_dataset = load_dataset(CXR_DATASET, 'findings_section', split='test')

In [ ]:
def attach_cxrs(history: dict) -> dict:
    sid = str(history['subject_id'])
    cxr_list = cxr_dataset.filter(lambda r, sid=sid: r['subject_id'] == sid)
    cxrs = []
    for r in cxr_list:
        image_filename = CXRS_DIR / f"{r['dicom_id']}.jpg"
        try:
            r['main_image'].save(image_filename)
        except Exception as e:
            print(f'skip {image_filename}: {e}')
            continue
        cxrs.append({
            'study_id': r.get('study_id'),
            'dicom_id': r.get('dicom_id'),
            'findings': r.get('findings_section'),
            'impression': r.get('impression_section'),
            'main_image_path': str(image_filename),
            'study_date': r.get('StudyDate'),
            'study_time': r.get('StudyTime'),
        })
    history['xray_studies'] = cxrs
    return history

In [ ]:
for sid in SUBJECT_IDS:
    history = build_history(sid)
    history = attach_cxrs(history)
    out_path = OUTPUT_DIR / f'patient_{sid}_history.json'
    with open(out_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'Wrote {out_path}: {len(history["admissions"])} admissions, {len(history["xray_studies"])} CXR studies')